# Needlet Filter Functions

Visualize the needlet filter functions used for the needlet ILC process. Needlets provide localized analysis in both angular position and scale, useful for foreground separation.

## Setup

In [ ]:
import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab

# Jupyter setup
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Custom plotting parameters
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Define Needlet Functions

In [ ]:
# Define angular multipoles
ls = np.arange(0, 4097)

@np.vectorize
def cos_h_I_l(l, input_I):
    """
    Cosine needlet filter function for scale I.
    
    Provides band-limited filters that sum to unity: Σ_I h_I(l)² = 1
    Each filter localizes power around specific angular scales.
    """
    I = input_I + 1
    
    # Define peak multipoles for each needlet scale
    l_peak_list = np.array([0, 0, 100, 200, 300, 400, 600, 800, 1000, 1250, 1400, 1800, 2200, 4097, 4097])
    
    # Cosine taper between adjacent peaks
    if l_peak_list[I-1] <= l < l_peak_list[I]:
        return np.cos(np.pi / 2 * (l_peak_list[I] - l) / (l_peak_list[I] - l_peak_list[I-1]))
    elif l_peak_list[I] <= l < l_peak_list[I+1]:
        return np.cos(np.pi /2 * (l - l_peak_list[I]) / (l_peak_list[I+1] - l_peak_list[I]))
    else:
        return 0

# Generate filter array for all scales
Nscales = 13
h_I_l_array = np.zeros((ls.shape[0], Nscales))

for l in ls:
    for I in range(Nscales):
        h_I_l_array[l, I] = cos_h_I_l(l, I)

## Visualize Filter Functions

In [ ]:
# Plot all needlet filters on log scale
for i in range(Nscales):
    plt.plot(ls, h_I_l_array[:, i])

plt.xscale("log")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$h_\ell^I$")
plt.savefig("plots/needlet_filters.pdf", bbox_inches='tight')